In [1]:
import pandas as pd
import polars as pl
import numpy as np
import os
from pathlib import Path
import pandas as pd
import re, pathlib

In [2]:
# =============================================================================
# 1.  Directory layout – pathlib all the way
# =============================================================================
SCRIPT_DIR   = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
PROJECT_ROOT = SCRIPT_DIR.parent          # edit if your notebook is elsewhere

DATA_DIR       = PROJECT_ROOT / "data/"
SIMULATION_DIR = DATA_DIR / "simulations/"          # folder with ATTRIBUTE_* and wide SSP file
TORNADO_SIM_DIR = SIMULATION_DIR /"data_for_LSU"
OUTPUT_DIR     = DATA_DIR / "output/"

In [3]:
louisiana = pd.read_csv(TORNADO_SIM_DIR / "louisiana.csv")

In [4]:
louisiana

,primary_id,region,time_period,area_agrc_crops_bevs_and_spices,area_agrc_crops_cereals,area_agrc_crops_fibers,area_agrc_crops_fruits,area_agrc_crops_herbs_and_other_perennial_crops,area_agrc_crops_nuts,area_agrc_crops_other_annual,...,emission_co2e_subsector_total_lndu,emission_co2e_subsector_total_lsmm,emission_co2e_subsector_total_lvst,emission_co2e_subsector_total_soil,emission_co2e_subsector_total_ccsq,emission_co2e_subsector_total_entc,emission_co2e_subsector_total_fgtv,emission_co2e_subsector_total_inen,emission_co2e_subsector_total_scoe,emission_co2e_subsector_total_trns
0,0,louisiana,0,0,367983.8318,68239.85746,80.234375,79198.54166,6714.566582,1154589.900,...,9.980000,0.322352,2.821843,4.032755,0.0,39.360000,15.368289,106.770000,4.290000,32.920000
1,0,louisiana,1,0,362385.6368,67201.71394,79.013757,77993.68198,6612.416842,1137024.945,...,9.968923,0.319496,2.800291,4.092947,0.0,40.166298,14.997723,108.605133,3.731796,31.374749
2,0,louisiana,2,0,361096.1267,66962.58391,78.732596,77716.14990,6588.887271,1132978.965,...,9.489254,0.320059,2.788854,4.099808,0.0,38.639666,13.763035,80.719306,3.559992,31.843131
3,0,louisiana,3,0,359935.7476,66747.40026,78.479589,77466.40976,6567.713942,1129338.147,...,10.279229,0.321689,2.777719,4.162110,0.0,41.465412,14.851001,146.300301,3.841408,33.650089
4,0,louisiana,4,0,358773.5212,66531.87404,78.226180,77216.27204,6546.506906,1125691.534,...,9.983308,0.323218,2.772568,4.173150,0.0,37.807488,13.488061,146.740554,3.879013,34.227053
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2407,127127,louisiana,31,0,311979.6878,53280.76885,83.099337,74341.21831,6871.576034,1116082.754,...,2.290044,0.099194,1.610628,2.566150,0.0,9.504129,1.357871,6.354095,0.494434,17.846785
2408,127127,louisiana,32,0,310670.5663,53049.87938,83.075690,74040.77745,6869.511396,1114713.368,...,2.285685,0.087899,1.527726,2.529939,0.0,9.783970,1.280190,4.725632,0.364936,17.059667
2409,127127,louisiana,33,0,309356.4650,52821.26235,83.037733,73734.23638,6866.309713,1113193.917,...,2.281655,0.076939,1.445208,2.497102,0.0,9.535228,1.030977,3.223424,0.239272,16.254621
2410,127127,louisiana,34,0,308044.9743,52594.98521,82.990728,73425.33726,6862.387927,1111581.984,...,2.277804,0.066352,1.363483,2.466951,0.0,9.696216,0.918047,1.841394,0.117569,15.427843


In [5]:
# 1) Filter out the base case
base_case = louisiana[louisiana['primary_id'] == 0].copy()


In [6]:
# 2) Define your fuels and sectors
relevant_fuels = ['solid_biomass', 
                    'coal',  
                    'diesel', 
                    'electricity',
                    'gasoline', 
                    'hydrocarbon_gas_liquids',
                    'hydrogen',
                    'kerosene',
                    'natural_gas']

sectors = ['commercial_municipal',
            'other_se',
            'residential']



In [7]:
# 3) Initialize accumulators
fuel_demand_by_sector = pd.DataFrame({'time_period': base_case['time_period']}, index=base_case.index)

In [8]:
# 4) Industrial cost parameters
capex_industrial_electricity = 92666.6 * 21
capex_industrial_other       = 92666.6 * 12
opex_industrial_electricity  = 92666.6 * 2.5
opex_industrial_other        = 92666.6 * 4.5
capex_multiplier_efficiency = 5560000
opex_multiplier_efficiency = 0

In [9]:
# 5) Loop over fuels and sectors
for fuel in relevant_fuels:
    # find the efficiency column(s) for this fuel
    # find the demand column(s) for this fuel
    for sector in sectors:
        eff_cols = [c for c in base_case.columns
                if c.startswith(f'efficfactor_scoe_heat_energy_{sector}_{fuel}')]
        fuel_efficiency = base_case[eff_cols[0]]
        sector_dem_cols = [c for c in base_case.columns
                if (f'scalar_scoe_heat_energy_demand_{sector}' in c)]
        
        sector_fuel_fraction_cols = [c for c in base_case.columns
                if (f'frac_scoe_heat_energy_{sector}_{fuel}' in c)]

        if len(sector_fuel_fraction_cols)>0 and len(sector_dem_cols)>0:
            sector_total_demand = base_case[sector_dem_cols[0]]
            sector_fuel_fraction = base_case[sector_fuel_fraction_cols[0]]

            if (sector_fuel_fraction*sector_total_demand).sum()>0 or fuel=='electricity':
                sector_fuel_demand = sector_fuel_fraction*sector_total_demand
                fuel_demand_by_sector[f'energy_demand_{sector}_{fuel}'] = sector_fuel_demand
                if fuel=='electricity':
                    fuel_demand_by_sector[f'energy_demand_capex_{sector}_{fuel}'] = sector_fuel_demand*capex_industrial_electricity
                    fuel_demand_by_sector[f'energy_demand_opex_{sector}_{fuel}'] = sector_fuel_demand*opex_industrial_electricity
                else:
                    fuel_demand_by_sector[f'energy_demand_capex_{sector}_{fuel}'] = sector_fuel_demand*capex_industrial_other
                    fuel_demand_by_sector[f'energy_demand_opex_{sector}_{fuel}'] = sector_fuel_demand*opex_industrial_other
                sector_fuel_consumed = sector_fuel_demand/fuel_efficiency
                sector_fuel_consumed_baseline = sector_fuel_demand/fuel_efficiency.iloc[0]
                sector_change_in_fuel_consumed = sector_fuel_consumed_baseline-sector_fuel_consumed
                fuel_demand_by_sector[f'efficiency_energy_saving_{sector}_{fuel}'] = sector_change_in_fuel_consumed
                fuel_demand_by_sector[f'efficiency_capex_{sector}_{fuel}'] = sector_change_in_fuel_consumed*capex_multiplier_efficiency
                fuel_demand_by_sector[f'efficiency_opex_{sector}_{fuel}'] = sector_change_in_fuel_consumed*opex_multiplier_efficiency


    


['scalar_scoe_heat_energy_demand_commercial_municipal']
['scalar_scoe_heat_energy_demand_other_se']
['scalar_scoe_heat_energy_demand_residential']
['scalar_scoe_heat_energy_demand_commercial_municipal']
['scalar_scoe_heat_energy_demand_other_se']
['scalar_scoe_heat_energy_demand_residential']
['scalar_scoe_heat_energy_demand_commercial_municipal']
['scalar_scoe_heat_energy_demand_other_se']
['scalar_scoe_heat_energy_demand_residential']
['scalar_scoe_heat_energy_demand_commercial_municipal']
['scalar_scoe_heat_energy_demand_other_se']
['scalar_scoe_heat_energy_demand_residential']
['scalar_scoe_heat_energy_demand_commercial_municipal']
['scalar_scoe_heat_energy_demand_other_se']
['scalar_scoe_heat_energy_demand_residential']
['scalar_scoe_heat_energy_demand_commercial_municipal']
['scalar_scoe_heat_energy_demand_other_se']
['scalar_scoe_heat_energy_demand_residential']
['scalar_scoe_heat_energy_demand_commercial_municipal']
['scalar_scoe_heat_energy_demand_other_se']
['scalar_scoe_heat

C:\Users\pkane\AppData\Local\Temp\ipykernel_12724\4224841512.py:31: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  fuel_demand_by_sector[f'efficiency_energy_saving_{sector}_{fuel}'] = sector_change_in_fuel_consumed
C:\Users\pkane\AppData\Local\Temp\ipykernel_12724\4224841512.py:32: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  fuel_demand_by_sector[f'efficiency_capex_{sector}_{fuel}'] = sector_change_in_fuel_consumed*capex_multiplier_efficiency
C:\Users\pkane\AppData\Local\Temp\ipykernel_12724\4224841512.py:33: PerformanceWarning

In [10]:
fuel_demand_by_sector

,time_period,energy_demand_commercial_municipal_solid_biomass,energy_demand_capex_commercial_municipal_solid_biomass,energy_demand_opex_commercial_municipal_solid_biomass,efficiency_energy_saving_commercial_municipal_solid_biomass,efficiency_capex_commercial_municipal_solid_biomass,efficiency_opex_commercial_municipal_solid_biomass,energy_demand_residential_solid_biomass,energy_demand_capex_residential_solid_biomass,energy_demand_opex_residential_solid_biomass,...,energy_demand_opex_other_se_natural_gas,efficiency_energy_saving_other_se_natural_gas,efficiency_capex_other_se_natural_gas,efficiency_opex_other_se_natural_gas,energy_demand_residential_natural_gas,energy_demand_capex_residential_natural_gas,energy_demand_opex_residential_natural_gas,efficiency_energy_saving_residential_natural_gas,efficiency_capex_residential_natural_gas,efficiency_opex_residential_natural_gas
0,0,0.000804,894.249741,335.343653,0.0,0.0,0.0,0.005405,6010.806036,2254.052263,...,41400.288162,0.0,0.0,0.0,0.254730,283259.255976,106222.220991,0.0,0.0,0.0
1,1,0.000811,901.498863,338.062074,0.0,0.0,0.0,0.003597,3999.996786,1499.998795,...,48468.402636,0.0,0.0,0.0,0.230935,256799.815476,96299.930803,0.0,0.0,0.0
2,2,0.000822,914.096702,342.786263,0.0,0.0,0.0,0.003014,3351.919205,1256.969702,...,58269.855067,0.0,0.0,0.0,0.223813,248880.001494,93330.000560,0.0,0.0,0.0
3,3,0.000778,865.031962,324.386986,0.0,0.0,0.0,0.003989,4436.167353,1663.562757,...,56140.761768,0.0,0.0,0.0,0.256649,285393.411513,107022.529317,0.0,0.0,0.0
4,4,0.000797,886.407922,332.402971,0.0,0.0,0.0,0.004138,4601.375962,1725.515986,...,53886.558134,0.0,0.0,0.0,0.253103,281450.831693,105544.061885,0.0,0.0,0.0
5,5,0.000859,954.915969,358.093488,0.0,0.0,0.0,0.003592,3994.249974,1497.843740,...,25258.038372,0.0,0.0,0.0,0.234914,261223.949885,97958.981207,0.0,0.0,0.0
6,6,0.000840,934.060424,350.272659,0.0,0.0,0.0,0.003484,3874.562165,1452.960812,...,40144.916820,0.0,0.0,0.0,0.257840,286717.563480,107519.086305,0.0,0.0,0.0
7,7,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.003484,3874.562165,1452.960812,...,40144.916820,0.0,0.0,0.0,0.257840,286717.563480,107519.086305,0.0,0.0,0.0
8,8,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.003484,3874.562165,1452.960812,...,40144.916820,0.0,0.0,0.0,0.257840,286717.563480,107519.086305,0.0,0.0,0.0
9,9,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.003484,3874.562165,1452.960812,...,40144.916820,0.0,0.0,0.0,0.257840,286717.563480,107519.086305,0.0,0.0,0.0


In [11]:


# 6) Write out to CSV

fuel_demand_by_sector.to_csv(OUTPUT_DIR/'scoe.csv', index=False)
